# Sales map

An example notebook for `examples/`: a map of a made-up chain of stores in Java, drawn with matplotlib alone (runs have no network, so no map tiles). Province shapes are Natural Earth's (public domain), simplified, in `data/java_provinces.geojson`; sales come from `data/orders.csv`.

- Provinces are shaded by `metric`: `revenue`, `units`, or `growth` (last 4 weeks against the first 4, in %).
- Each store is a bubble: its size is its revenue, its colour its growth.

The map is kept as `map.png`, and the provinces with their figures as `provinces.geojson`, ready for a GIS tool.

In [ ]:
# Parameters: the caller's values replace these.
metric = "revenue"
weeks = 13

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

import nbsandbox

OUT = os.environ["NBSANDBOX_OUTPUT_DIR"]  # files written here become the run's artifacts
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})


def save(fig, name):
    """Show a figure in the notebook and keep it as a PNG artifact."""
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight")
    plt.show()


orders = pd.read_csv("data/orders.csv", parse_dates=["week"])
stores = pd.read_csv("data/stores.csv")
sales = orders.merge(stores, on="store")

import math

import matplotlib.patheffects as pe
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import TwoSlopeNorm

if metric not in ("revenue", "units", "growth"):
    raise ValueError("metric must be revenue, units or growth")
first = sales["week"].max() - pd.Timedelta(weeks=weeks - 1)
sales = sales[sales["week"] >= first]

with open("data/java_provinces.geojson") as f:
    provinces = json.load(f)["features"]
print(f"{len(provinces)} provinces, {sales['store'].nunique()} stores, {sales['week'].nunique()} weeks")

## Figures per province and per store

In [ ]:
def growth_of(frame, key):
    w = frame.groupby([key, "week"])["revenue"].sum().unstack()
    return (w.iloc[:, -4:].sum(axis=1) / w.iloc[:, :4].sum(axis=1) - 1) * 100

prov = sales.groupby("province")[["revenue", "units"]].sum()
prov["revenue"] /= 1000  # IDR millions
prov["growth"] = growth_of(sales, "province")
store = sales.groupby(["store", "city", "lat", "lon"])["revenue"].sum().div(1000).reset_index().set_index("store")
store["growth"] = growth_of(sales, "store")
prov.round(1)

## The map

An equirectangular projection, stretched by 1/cos(latitude) so shapes keep their proportions this close to the equator.

In [ ]:
def rings(feature):
    for polygon in feature["geometry"]["coordinates"]:
        yield np.array(polygon[0])

lat0 = math.radians(-7.3)
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.set_aspect(1 / math.cos(lat0))

values = [prov[metric].get(f["properties"]["province"], np.nan) for f in provinces]
polys, colors = [], []
for f, v in zip(provinces, values):
    for r in rings(f):
        polys.append(r)
        colors.append(v)
if metric == "growth":
    norm = TwoSlopeNorm(vcenter=0, vmin=min(-1, np.nanmin(colors)), vmax=max(1, np.nanmax(colors)))
    cmap = "RdYlGn"
else:
    norm, cmap = plt.Normalize(np.nanmin(colors), np.nanmax(colors)), "YlGnBu"
pc = PolyCollection(polys, array=np.array(colors), cmap=cmap, norm=norm, edgecolors="white", linewidths=0.8)
ax.add_collection(pc)
unit = {"revenue": "IDR millions", "units": "units", "growth": "%"}[metric]
fig.colorbar(pc, ax=ax, shrink=0.7, pad=0.01, label=f"{metric} per province ({unit})")

# Province names, placed by hand where no store covers them. Jakarta is too
# small for its name: it gets a label out at sea, with a line to it.
names_at = {"Banten": (106.1, -6.65), "Jawa Barat": (107.95, -7.35), "Jawa Tengah": (109.95, -7.2),
            "DI Yogyakarta": (110.4, -8.1), "Jawa Timur": (111.85, -7.7)}
halo = [pe.withStroke(linewidth=2.5, foreground="white")]  # readable on any shade
for name, (x, y) in names_at.items():
    ax.text(x, y, name, ha="center", fontsize=7.5, color="#333", style="italic", zorder=5, path_effects=halo)
ax.annotate("DKI Jakarta", xy=(106.85, -6.12), xytext=(107.25, -5.78), fontsize=7.5, color="#333", style="italic",
            arrowprops={"arrowstyle": "-", "color": "#666", "linewidth": 0.6}, zorder=5, path_effects=halo)

# Stores: size = revenue, colour = growth.
gnorm = TwoSlopeNorm(vcenter=0, vmin=min(-1, store["growth"].min()), vmax=max(1, store["growth"].max()))
size = 900 * store["revenue"] / store["revenue"].max()
sc = ax.scatter(store["lon"], store["lat"], s=size, c=store["growth"], cmap="RdYlGn", norm=gnorm,
                edgecolors="black", linewidths=0.6, alpha=0.9, zorder=3)
# Store names beside their bubbles; the crowded ones around Jakarta moved apart.
offsets = {"Tangerang": (-12, -14, "right"), "Jakarta Pusat": (10, 10, "left"), "Jakarta Selatan": (12, -6, "left"),
           "Bogor": (12, -6, "left"), "Serang": (-8, 8, "right"), "Banyuwangi": (-8, 8, "right")}
for s, row in store.iterrows():
    dx, dy, ha = offsets.get(row["city"], (6, 6, "left"))
    ax.annotate(row["city"], (row["lon"], row["lat"]), xytext=(dx, dy), textcoords="offset points", ha=ha, fontsize=7, zorder=4,
                path_effects=halo)

# Bubble size legend.
for v in (store["revenue"].max(), store["revenue"].max() / 4):
    ax.scatter([], [], s=900 * v / store["revenue"].max(), facecolors="none", edgecolors="black", label=f"{v:,.0f} M")
ax.legend(title="store revenue", loc="lower left", frameon=False, labelspacing=1.4, borderpad=1)

# A 100 km scale bar and a north arrow.
km_per_deg = 111.32 * math.cos(lat0)
x0, y0 = 113.6, -6.25
ax.plot([x0, x0 + 100 / km_per_deg], [y0, y0], color="black", linewidth=2)
ax.text(x0 + 50 / km_per_deg, y0 + 0.07, "100 km", ha="center", fontsize=7.5)
ax.annotate("N", xy=(114.3, -5.75), xytext=(114.3, -6.15), ha="center", fontsize=9,
            arrowprops={"arrowstyle": "-|>", "color": "black"})

ax.set_xlim(105.0, 114.8)
ax.set_ylim(-8.9, -5.6)
ax.grid(False)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"Stores and provinces of Java: {metric}, last {sales['week'].nunique()} weeks")
save(fig, "map.png")

## The result

`provinces.geojson` carries each province's figures, for QGIS or any GIS tool; the caller gets them as JSON.

In [ ]:
for f in provinces:
    p = f["properties"]["province"]
    if p in prov.index:
        f["properties"].update({k: round(float(prov.loc[p, k]), 1) for k in ("revenue", "units", "growth")})
with open(os.path.join(OUT, "provinces.geojson"), "w") as fh:
    json.dump({"type": "FeatureCollection", "features": provinces}, fh)

nbsandbox.output({
    "metric": metric,
    "weeks": int(sales["week"].nunique()),
    "provinces": prov.round(1).reset_index(),
    "stores": store.round(1).reset_index()[["store", "city", "revenue", "growth"]],
})